# Sıfırdan Dil Modeli · Bölüm 8
## GPT'nin parçaları

Önceki bölümlerde metni parçalara böldük, parçaları vektörlere çevirdik ve dikkati (maske, çok başlı dikkat) yazdık. Ama bir GPT bloğunda dikkat yalnız bir parça. Bu defterde modelin iskeletini kuruyor ve bloğun öteki parçalarını yazıyoruz: **katman normalleştirme** (*LayerNorm*), **GELU**, **ileri beslemeli ağ** (*FeedForward*) ve **kısa yol bağlantıları** (*shortcut connections*).

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 4.1–4.4. bölümlerini izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Ayarlar
Modelin bütün boyutları tek bir sözlükte. Kitap burada GPT-2'nin en küçük sürümünün ayarlarını kullanır (124 milyon parametre). Biz serinin modeli **Çırak**'ın ayarlarını kullanıyoruz: E13'te Çırak'ın eğitilmiş ağırlıklarını bu modele yükleyeceğiz, boyutların birebir tutması gerekiyor.

In [ ]:
import torch
import torch.nn as nn

GPT_CONFIG_CIRAK = {
    "vocab_size": 4096,     # sözlük: Çırak'ın tokenizer'ı
    "context_length": 256,  # bağlam: en fazla kaç token
    "emb_dim": 256,         # gömme: her token kaç sayı
    "n_heads": 8,           # dikkat başlığı sayısı
    "n_layers": 6,          # transformer bloğu sayısı
    "drop_rate": 0.1,       # dropout oranı
    "qkv_bias": True        # sorgu, anahtar, değerde bias
}

## 2. İskelet
Önce modelin iskeleti: girişte token ve konum gömmeleri (Bölüm 4), ortada art arda `n_layers` tane transformer bloğu, sonda bir normalleştirme ve sözlüğün her parçasına bir puan veren çıkış katmanı. Bloklar ve normalleştirme şimdilik **yer tutucu**: hiçbir şey yapmadan girdiyi aynen geri veriyorlar. `nn.Sequential` katmanları art arda bağlar; `*[...]` listeyi tek tek argümanlara açar.

In [ ]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # yer tutucu: transformer blokları
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        # yer tutucu: LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x    # şimdilik hiçbir şey yapmıyor


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x    # şimdilik hiçbir şey yapmıyor

Modeli denemek için iki kısa metin lazım. Önceki bölümlerin kodunu (`onceki_bolumler.py`) ve Çırak'ın tokenizer dosyasını serinin deposundan indiriyoruz:

In [ ]:
import os
import urllib.request

repo = ("https://raw.githubusercontent.com/"
        "kocahmet1/sifirdan-dil-modeli/main/")
for path in ["onceki_bolumler.py", "veri/cirak_tokenizer.json"]:
    file_name = os.path.basename(path)
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(repo + path, file_name)
    print(file_name, os.path.getsize(file_name), "bayt")

Kaşağı'dan iki kısa metin; ikisi de dört parça. `torch.stack` iki satırı tek bir tensörde üst üste koyar: bir *batch*.

In [ ]:
from onceki_bolumler import cirak_tokenizer

tokenizer = cirak_tokenizer()

batch = []
txt1 = "Ben bir gün"
txt2 = "Ben o gün"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

İskeleti kurup bu batch'i içinden geçiriyoruz. Çıktı her token için sözlükteki her parçaya bir puan: **logit**.

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_CIRAK)

logits = model(batch)
print("Çıktının şekli:", logits.shape)
print(logits)

## 3. Normalleştirme
Derin bir ağda sayılar katmandan katmana kayar, büyür ya da küçülür; eğitim zorlaşır. **Katman normalleştirme** her satırı ortalaması 0, varyansı 1 olacak şekilde yeniden ölçekler. Önce elle: beş sayılık iki örnek, bir doğrusal katman ve ReLU.

In [ ]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

Her satırın ortalaması ve varyansı. `dim=-1` son boyut boyunca (satır satır), `keepdim=True` sonucu sütun biçiminde tutar.

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("Ortalama:\n", mean)
print("Varyans:\n", var)

Normalleştirmek: ortalamayı çıkar, standart sapmaya (varyansın karekökü) böl.

In [ ]:
torch.set_printoptions(sci_mode=False)   # çok küçük sayılar 0.0000 görünsün

out_norm = (out - mean) / torch.sqrt(var)
print("Normalleştirilmiş:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Ortalama:\n", mean)
print("Varyans:\n", var)

## 4. LayerNorm sınıfı
Aynı işi bir katman olarak yazıyoruz. `eps` sıfıra bölmeyi önleyen küçük bir sayı. `scale` ve `shift` öğrenilen iki parametre (`nn.Parameter`): eğitim, normalleştirmeyi gerekirse geri alabilsin diye. `unbiased=False`: varyansı n'ye bölerek hesapla (GPT-2'deki gibi).

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, keepdim=True, unbiased=False)
print("Ortalama:\n", mean)
print("Varyans:\n", var)

## 5. GELU
Aktivasyon fonksiyonu, sinir ağına eğrilik katar. Klasik olanı ReLU: eksileri sıfırlar, artıları olduğu gibi bırakır. GPT-2 bunun yumuşak bir hâlini kullanır: **GELU**. Aşağıdaki, kitapta da kullanılan tanh yaklaşımı.

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

İkisini yan yana çizelim. `torch.linspace(-3, 3, 100)`: -3 ile 3 arasında eşit aralıklı 100 sayı.

In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} aktivasyon fonksiyonu")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

Birkaç sayıda karşılaştıralım. Üçüncü satır PyTorch'un kendi GELU'su, yaklaşımsız (tam) hâli; Çırak bununla eğitildi.

In [ ]:
xs = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("GELU:    ", gelu(xs))
print("ReLU:    ", relu(xs))
print("tam GELU:", nn.functional.gelu(xs))

## 6. İleri beslemeli ağ (FeedForward)
İki doğrusal katman, aralarında GELU. Önce her token'ın 256 sayısını dört katına (1.024) çıkarır, sonra yeniden 256'ya indirir. Her token ayrı ayrı geçer: burada tokenlar birbirine bakmaz, o iş dikkatindi.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
ffn = FeedForward(GPT_CONFIG_CIRAK)
x = torch.rand(2, 3, 256)    # 2 metin, 3'er token, token başına 256 sayı
out = ffn(x)
print(out.shape)

Bu küçük sınıfın kaç parametresi var? `numel` bir tensördeki sayı adedi. Modelde her blokta bir tane var; Çırak'ta 6 blok.

In [ ]:
n_params = sum(p.numel() for p in ffn.parameters())
print("Bir blokta:", n_params)
print("6 blokta:  ", n_params * GPT_CONFIG_CIRAK["n_layers"])

## 7. Kısa yol bağlantıları
Derin ağlarda geriye doğru giden gradyanlar katman katman küçülür; ilk katmanlar neredeyse hiç öğrenemez. **Kısa yol** (*shortcut* ya da *residual*) bağlantısı bir katmanın girdisini çıktısına ekler: `x = x + layer_output`. Beş katmanlı küçük bir ağla deneyelim. Liste kurucusu (`[... for i in ...]`) her ardışık boyut çifti için bir katman kurar.

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i + 1]), GELU())
            for i in range(len(layer_sizes) - 1)
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output    # kısa yol: girdiyi çıktıya ekle
            else:
                x = layer_output
        return x

Bir ileri geçiş, hedefe göre bir kayıp (`MSELoss`: ortalama kare hata), sonra `backward` (Ara Bölüm A'daki autograd). Her katmanın ağırlıklarına gelen gradyanın ortalama büyüklüğünü yazdırıyoruz.

In [ ]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)

    loss.backward()

    for name, param in model.named_parameters():
        if "weight" in name:
            print(f"{name}: {param.grad.abs().mean().item():.6f}")

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

## Özet
Bir GPT'nin iskeleti: gömmeler → `n_layers` transformer bloğu → son normalleştirme → çıkış katmanı. Bugün bloğun dikkat dışındaki parçalarını yazdık: her satırı ortalaması 0, varyansı 1 olacak şekilde ölçekleyen **LayerNorm**, yumuşak bir aktivasyon olan **GELU**, token başına çalışan ve modelin parametrelerinin yarısından fazlasını taşıyan **FeedForward**, ve gradyanın ilk katmanlara kadar ulaşmasını sağlayan **kısa yol bağlantıları**.

**Kendin dene**
- `layer_sizes = [3] * 10 + [1]` ile on katmanlı ağı deneyin: kısa yolsuz ve kısa yollu modelin ilk katman gradyanları ne oluyor?
- `GPT_CONFIG_CIRAK["emb_dim"]`'i 512 yapıp `FeedForward`'ın parametrelerini yeniden sayın: kaç kat büyüyor, neden?

**Sıradaki bölüm:** GPT'yi birleştirmek: transformer bloğu, `GPTModel` ve modelin ilk (anlamsız) Türkçesi.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 4.1–4.4. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Örnek metinler: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.